# organizar-efeitos-audio.ipynb — Organiza o estoque de efeitos sonoros

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Igual `organizar-trilha-audio.ipynb`, só que pra **efeito sonoro** (porta, trovão, espada, cavalo...) em vez de trilha por clima:

1. **Freesound_Audio_Manager** (a mesma planilha da trilha) -- filtra por tag **concreta/visual**, não clima
2. Varre uma pasta do Drive (ex: `Efeitos/`) com subpastas por **objeto/ação** (`Efeitos/porta/`, `Efeitos/trovao/`, `Efeitos/cavalo/`...) -- não por clima
3. Combina tudo na MESMA aba `trilha_stock` de `Biblioteca_Match_Audio` (estoque único de som -- trilha e efeito juntos, diferenciados pela coluna `categoria`)

**Primeira vez**: se a pasta de Efeitos ainda não existir no Drive, crie ela com subpastas por objeto/ação antes de rodar.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SETUP                                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U gspread google-api-python-client

from google.colab import drive, auth
from google.auth import default
import gspread
from googleapiclient.discovery import build

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

import shutil
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
_pasta_modulos = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos"
for _arquivo in Path(_pasta_modulos).glob("*.py"):
    shutil.copy(_arquivo, ".")

print("✅ Setup pronto")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup pronto


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

ID_BIBLIOTECA_MATCH_AUDIO = "1VkYaApN1F7X4-52CD0_I-TpKwc2v3XuhUJrIRnZ0Q94"  # a mesma de trilha

# Reusa a mesma Freesound_Audio_Manager da trilha (filtra por tag concreta
# em vez de clima) -- deixe em branco se não quiser puxar de lá pra efeito
ID_PLANILHA_FREESOUND_MANAGER = "1ieROA_Yy_1fM_qZ_uweLycEJ81l6LwZVJzbYj-sAUA4"
NOME_ABA_FREESOUND_MANAGER = "Página1"

# Caminho da pasta de EFEITOS no Drive (com subpastas por objeto/ação --
# ex: Efeitos/porta/, Efeitos/trovao/, Efeitos/cavalo/) -- ajuste conforme
# onde você organizar isso
CAMINHO_PASTA_EFEITOS_DRIVE = "Dark NPOS AlanaBdMorais/Efeitos"

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Biblioteca_Match_Audio: {ID_BIBLIOTECA_MATCH_AUDIO}")
print(f"   Freesound (opcional):   {ID_PLANILHA_FREESOUND_MANAGER or '(pulando)'}")
print(f"   Pasta de efeitos:       {CAMINHO_PASTA_EFEITOS_DRIVE}")
print("=" * 60)

⚙️  CONFIGURAÇÃO
   Biblioteca_Match_Audio: 1VkYaApN1F7X4-52CD0_I-TpKwc2v3XuhUJrIRnZ0Q94
   Freesound (opcional):   1ieROA_Yy_1fM_qZ_uweLycEJ81l6LwZVJzbYj-sAUA4
   Pasta de efeitos:       Dark NPOS AlanaBdMorais/Efeitos


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  ABRIR A BIBLIOTECA_MATCH_AUDIO                               ║
# ╚══════════════════════════════════════════════════════════════════╝
from trilha_pipeline import (
    carregar_efeitos_stock_freesound, carregar_efeitos_stock_pasta_drive,
    garantir_aba_estoque_som, sincronizar_estoque_som, achar_pasta_por_caminho,
)

spreadsheet_biblioteca_audio = gc.open_by_key(ID_BIBLIOTECA_MATCH_AUDIO)
aba_efeitos_stock = garantir_aba_estoque_som(spreadsheet_biblioteca_audio, "trilha_stock")
print("✅ Aba trilha_stock (estoque único) aberta/criada")

✅ Aba efeitos_stock aberta/criada


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3️⃣  FREESOUND (opcional) — filtra por tag CONCRETA               ║
# ╚══════════════════════════════════════════════════════════════════╝
stock_freesound_efeitos = []
if ID_PLANILHA_FREESOUND_MANAGER:
    _aba_freesound_manager = gc.open_by_key(ID_PLANILHA_FREESOUND_MANAGER).worksheet(NOME_ABA_FREESOUND_MANAGER)
    linhas_freesound = _aba_freesound_manager.get_all_records()
    stock_freesound_efeitos = carregar_efeitos_stock_freesound(linhas_freesound)
    print(f"📋 Freesound: {len(stock_freesound_efeitos)} com tag concreta identificada (de {len(linhas_freesound)} linha(s))")
else:
    print("⏭️  Pulando Freesound (ID não configurado)")

📋 Freesound: 0 com tag concreta identificada (de 20 linha(s))


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4️⃣  DRIVE — varre a pasta de efeitos (subpasta = objeto/ação)    ║
# ╚══════════════════════════════════════════════════════════════════╝
_pasta_raiz_id = achar_pasta_por_caminho(drive_service, CAMINHO_PASTA_EFEITOS_DRIVE)
print(f"📁 Pasta '{CAMINHO_PASTA_EFEITOS_DRIVE}' encontrada (id: {_pasta_raiz_id})")

stock_drive_efeitos = carregar_efeitos_stock_pasta_drive(drive_service, _pasta_raiz_id, tornar_publico=True)
print(f"\n📋 {len(stock_drive_efeitos)} arquivo(s) de efeito encontrados no total")

📁 Pasta 'Dark NPOS AlanaBdMorais/Efeitos' encontrada (id: 1yoXRg7d3OnhMjyl0sKDMa__hAHyFzd6Q)
   📁 1 subpasta(s) de efeito encontrada(s): Suspense
   📂 Suspense: 1 arquivo(s)

📋 1 arquivo(s) de efeito encontrados no total


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5️⃣  SINCRONIZAR — estoque único (trilha_stock)                  ║
# ╚══════════════════════════════════════════════════════════════════╝
efeitos_combinado = stock_freesound_efeitos + stock_drive_efeitos
n_novas, n_atualizadas = sincronizar_estoque_som(aba_efeitos_stock, efeitos_combinado)
print(f"✅ Estoque único (trilha_stock): {n_novas} nova(s), {n_atualizadas} atualizada(s)")
print(f"\n📊 Total: {len(efeitos_combinado)} efeito(s) disponível(is) pro painel de revisão")

✅ efeitos_stock: 1 nova(s), 0 atualizada(s)

📊 Total: 1 efeito(s) disponível(is) pro painel de revisão
